# UpLift — Notebook 01: Data Generation & Exploratory Analysis
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

This notebook generates a realistic synthetic dataset that mirrors the structure of publicly available transit outage records (MTA NYC, SEPTA, WMATA) and NOAA weather data. It then performs exploratory data analysis (EDA) to surface patterns that motivate the UpLift model design.

**Real data sources** (for production use):
- [MTA Elevator & Escalator Availability](https://data.ny.gov/Transportation/MTA-Subway-Elevator-and-Escalator-Availability-Beg/rc78-7x78)
- [SEPTA Elevator Outages API](https://opendataphilly.org/datasets/septa-elevator-outages/)
- [NOAA Climate Data Online](https://www.ncei.noaa.gov/cdo-web/)
- [NTD Breakdown Records 2022–2024](https://datahub.transportation.gov/Public-Transit/2022-2024-NTD-Annual-Data-Breakdowns/amkt-4ehs)

> **Note:** This notebook uses synthetic data that mirrors real-world distributions. Replace `df` with real agency data once available.

In [ ]:
# Install dependencies (run once in Google Colab)
# Uncomment the line below if running in Colab
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

## 1. Generate Synthetic Equipment Dataset

In [ ]:
N = 2000  # number of equipment-window records

equipment_types = ['elevator', 'escalator', 'platform_lift']
agencies = ['SEPTA', 'MTA_NYC', 'WMATA', 'CTA', 'MBTA']
manufacturers = ['Otis', 'Schindler', 'KONE', 'ThyssenKrupp', 'Mitsubishi']

df = pd.DataFrame({
    'equipment_id': [f'EQ-{str(i).zfill(4)}' for i in range(N)],
    'equipment_type': np.random.choice(equipment_types, N, p=[0.5, 0.35, 0.15]),
    'agency': np.random.choice(agencies, N, p=[0.3, 0.35, 0.15, 0.1, 0.1]),
    'manufacturer': np.random.choice(manufacturers, N),
    'equipment_age_years': np.random.gamma(shape=3, scale=5, size=N).clip(0.5, 40),
    'days_since_last_maintenance': np.random.exponential(scale=60, size=N).clip(1, 365),
    'unplanned_outages_12mo': np.random.negative_binomial(2, 0.4, N),
    'total_usage_load_millions': np.random.gamma(shape=2, scale=1.5, size=N).clip(0.1, 15),
    'days_since_parts_replacement': np.random.exponential(scale=180, size=N).clip(1, 730),
    'inspection_score': np.random.normal(loc=78, scale=12, size=N).clip(20, 100),
    'avg_monthly_temp_f': np.random.normal(loc=58, scale=18, size=N).clip(10, 100),
    'avg_daily_ridership': np.random.gamma(shape=3, scale=8000, size=N).clip(500, 80000),
    'floors_served': np.random.choice([2, 3, 4, 5, 6], N, p=[0.3, 0.35, 0.2, 0.1, 0.05]),
    'is_transfer_hub': np.random.choice([0, 1], N, p=[0.7, 0.3]),
})

# Generate label: outage_within_30_days
# Built-in class imbalance (~25% positive) — realistic for maintenance prediction
log_odds = (
    -3.5
    + 0.08 * df['equipment_age_years']
    + 0.005 * df['days_since_last_maintenance']
    + 0.25 * df['unplanned_outages_12mo']
    - 0.02 * df['inspection_score']
    + 0.003 * df['days_since_parts_replacement']
    + 0.01 * df['total_usage_load_millions']
    + 0.5 * df['is_transfer_hub']
    + np.where(df['equipment_type'] == 'escalator', 0.4, 0)
    + np.where(df['equipment_type'] == 'platform_lift', 0.6, 0)
    + np.random.normal(0, 0.5, N)
)
prob = 1 / (1 + np.exp(-log_odds))
df['outage_within_30_days'] = (np.random.random(N) < prob).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'\nClass balance:')
print(df['outage_within_30_days'].value_counts(normalize=True).round(3))
df.head()

## 2. Save Dataset

In [ ]:
df.to_csv('uplift_equipment_data.csv', index=False)
print('Saved: uplift_equipment_data.csv')
print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('UpLift — Equipment Outage Risk: Exploratory Analysis', 
             fontsize=15, fontweight='bold', y=1.01)

# 1. Outage rate by equipment type
outage_by_type = df.groupby('equipment_type')['outage_within_30_days'].mean() * 100
outage_by_type.sort_values(ascending=True).plot(
    kind='barh', ax=axes[0,0], color=['#2196F3','#FF9800','#F44336'])
axes[0,0].set_title('Outage Rate by Equipment Type', fontweight='bold')
axes[0,0].set_xlabel('30-Day Outage Rate (%)')
axes[0,0].xaxis.set_major_formatter(mtick.PercentFormatter())

# 2. Outage rate by agency
outage_by_agency = df.groupby('agency')['outage_within_30_days'].mean() * 100
outage_by_agency.sort_values(ascending=True).plot(
    kind='barh', ax=axes[0,1], color='#5C6BC0')
axes[0,1].set_title('Outage Rate by Agency', fontweight='bold')
axes[0,1].set_xlabel('30-Day Outage Rate (%)')
axes[0,1].xaxis.set_major_formatter(mtick.PercentFormatter())

# 3. Equipment age vs outage
df.boxplot(column='equipment_age_years', by='outage_within_30_days', ax=axes[0,2])
axes[0,2].set_title('Equipment Age vs Outage', fontweight='bold')
axes[0,2].set_xlabel('Outage in 30 Days (0=No, 1=Yes)')
axes[0,2].set_ylabel('Equipment Age (Years)')
plt.sca(axes[0,2])
plt.title('Equipment Age vs Outage')

# 4. Prior outages histogram
df[df['outage_within_30_days']==1]['unplanned_outages_12mo'].plot(
    kind='hist', ax=axes[1,0], bins=15, alpha=0.7, color='#F44336', label='Will fail')
df[df['outage_within_30_days']==0]['unplanned_outages_12mo'].plot(
    kind='hist', ax=axes[1,0], bins=15, alpha=0.7, color='#4CAF50', label='Will not fail')
axes[1,0].set_title('Prior Outages (12 mo) Distribution', fontweight='bold')
axes[1,0].set_xlabel('# Prior Unplanned Outages')
axes[1,0].legend()

# 5. Inspection score vs outage
df.boxplot(column='inspection_score', by='outage_within_30_days', ax=axes[1,1])
axes[1,1].set_title('Inspection Score vs Outage', fontweight='bold')
axes[1,1].set_xlabel('Outage in 30 Days (0=No, 1=Yes)')
axes[1,1].set_ylabel('Inspection Score')
plt.sca(axes[1,1])
plt.title('Inspection Score vs Outage')

# 6. Transfer hub outage rate
hub_outage = df.groupby('is_transfer_hub')['outage_within_30_days'].mean() * 100
hub_outage.index = ['Non-Hub Station', 'Transfer Hub']
hub_outage.plot(kind='bar', ax=axes[1,2], color=['#81C784','#E53935'], rot=0)
axes[1,2].set_title('Outage Rate: Transfer Hubs vs Others', fontweight='bold')
axes[1,2].set_ylabel('30-Day Outage Rate (%)')
axes[1,2].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('uplift_eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_eda_charts.png')

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ['equipment_age_years', 'days_since_last_maintenance',
                'unplanned_outages_12mo', 'total_usage_load_millions',
                'days_since_parts_replacement', 'inspection_score',
                'avg_monthly_temp_f', 'avg_daily_ridership',
                'floors_served', 'is_transfer_hub', 'outage_within_30_days']

fig, ax = plt.subplots(figsize=(12, 9))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('UpLift Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('uplift_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_correlation_matrix.png')

## 4. Key Findings

| Finding | Implication for Model |
|---|---|
| Platform lifts have highest outage rate | `equipment_type` is a strong feature |
| Transfer hubs fail at higher rates | `is_transfer_hub` flag captures demand stress |
| Prior outages strongly predict future outages | `unplanned_outages_12mo` will rank high in SHAP |
| Inspection score inversely correlated with outage | Validates inspection data as useful signal |
| Equipment age shows clear separation | Older equipment = meaningfully higher risk |

➡️ **Next:** Notebook 02 — Preprocessing, SMOTE, and train/holdout split